# Task 1 — Classical Portfolio Benchmark

This notebook establishes the classical reference for Qiskit Fall Fest 2026 Track 2.

We select exactly **K = 3 stocks from n = 6** and minimize the risk-adjusted portfolio cost. We enumerate all valid portfolios so later QAOA results can be compared against a trustworthy classical baseline.

**Important:** This is only the classical benchmark. It does not implement QAOA yet.

## 1. Imports and reproducible dataset

The seed `2026` ensures that the same synthetic dataset is generated each time this notebook is run.

In [1]:
import numpy as np
from itertools import combinations

rng = np.random.default_rng(2026)

n, K, q = 6, 3, 0.5
mu = rng.uniform(0.05, 0.25, n)  # expected returns
A = rng.normal(0, 0.1, (n, n))
sigma = A @ A.T                  # covariance matrix (positive semidefinite)

print('Number of stocks:', n)
print('Stocks to select:', K)
print('Risk aversion q:', q)
print('Expected returns:', np.round(mu, 4))
print('Covariance matrix shape:', sigma.shape)

Number of stocks: 6
Stocks to select: 3
Risk aversion q: 0.5
Expected returns: [0.0858 0.178  0.1435 0.1241 0.121  0.2081]
Covariance matrix shape: (6, 6)


## 2. Define the portfolio cost

The challenge defines

$$C(x)=q x^T\Sigma x-\mu^T x+\lambda(\sum_i x_i-K)^2.$$

Here, `x[i] = 1` means buy stock `i`; `x[i] = 0` means do not buy it. The penalty discourages portfolios that do not select exactly `K` stocks. We use `lambda = 1.0` as a starting value only; the penalty sweep will study other values later.

In [2]:
def portfolio_cost(x, lam=1.0):
    """Return the challenge's risk-adjusted cost for a binary selection vector."""
    x = np.asarray(x, dtype=float)
    risk = q * (x @ sigma @ x)
    return_term = -(mu @ x)
    constraint_penalty = lam * (x.sum() - K) ** 2
    return risk + return_term + constraint_penalty

def selection_vector(indices):
    """Convert a tuple of selected zero-based stock indices into a binary vector."""
    x = np.zeros(n, dtype=int)
    x[list(indices)] = 1
    return x

example = selection_vector((0, 2, 4))
print('Example selection vector:', example)
print('Number selected:', example.sum())
print('Example cost:', round(portfolio_cost(example), 6))

Example selection vector: [1 0 1 0 1 0]
Number selected: 3
Example cost: -0.318189


## 3. Enumerate all valid portfolios

There are $\binom{6}{3}=20$ portfolios that select exactly three stocks. We test all of them and choose the one with the lowest cost. The penalty term is zero for every valid portfolio, so it does not change their relative ranking.

In [3]:
valid_portfolios = list(combinations(range(n), K))

results = []
for portfolio in valid_portfolios:
    x = selection_vector(portfolio)
    results.append({
        'portfolio_indices': portfolio,
        'selection_vector': ''.join(map(str, x)),
        'cost': portfolio_cost(x, lam=1.0),
        'risk_term': q * (x @ sigma @ x),
        'return_term': -(mu @ x),
    })

results = sorted(results, key=lambda row: row['cost'])
print('Valid portfolios evaluated:', len(results))
print('\nTop 5 valid portfolios:')
for row in results[:5]:
    print(row['selection_vector'], row['portfolio_indices'], f"cost={row['cost']:.6f}")

best = results[0]
print('\nBest valid portfolio:')
print('Zero-based stock indices:', best['portfolio_indices'])
print('Binary selection vector:', best['selection_vector'])
print('Minimum valid cost:', round(best['cost'], 8))

Valid portfolios evaluated: 20

Top 5 valid portfolios:
011001 (1, 2, 5) cost=-0.503015
010101 (1, 3, 5) cost=-0.478709
110001 (0, 1, 5) cost=-0.464474
010011 (1, 4, 5) cost=-0.459651
001101 (2, 3, 5) cost=-0.422080

Best valid portfolio:
Zero-based stock indices: (1, 2, 5)
Binary selection vector: 011001
Minimum valid cost: -0.50301469


## 4. Check the full bitstring space

The challenge's approximation-ratio formula uses the highest and lowest cost across **all 64 bitstrings**, including invalid portfolios. We calculate these separately from the valid-portfolio benchmark.

In [4]:
all_bitstrings = list(__import__('itertools').product([0, 1], repeat=n))
all_costs = np.array([portfolio_cost(bits, lam=1.0) for bits in all_bitstrings])

C_min = float(all_costs.min())
C_max = float(all_costs.max())
valid_count = sum(sum(bits) == K for bits in all_bitstrings)

print('Total bitstrings:', len(all_bitstrings))
print('Valid bitstrings selecting exactly K stocks:', valid_count)
print('C_min across all bitstrings:', C_min)
print('C_max across all bitstrings:', C_max)
assert len(valid_portfolios) == 20
assert len(all_bitstrings) == 64
assert valid_count == 20
print('All benchmark checks passed.')

Total bitstrings: 64
Valid bitstrings selecting exactly K stocks: 20
C_min across all bitstrings: -0.5030146902647753
C_max across all bitstrings: 9.0
All benchmark checks passed.


## 5. What to record

After running all cells, save these values for later QAOA comparisons:

- Best valid portfolio (indices and bitstring)
- Minimum valid portfolio cost
- `C_min` and `C_max` over all 64 bitstrings
- Confirmation that 20 valid portfolios and 64 total bitstrings were evaluated

**Do not claim QAOA performance from this notebook.** The next task will build the portfolio Hamiltonian and implement QAOA using the tutorial's workflow.